In [1]:
# predictor.py
!pip install segmentation-models-pytorch

import torch
import numpy as np
import cv2
import segmentation_models_pytorch as smp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256
NUM_CLASSES = 6  # 0 background + 5 classes

# SAME label map
label_map = {
    "short sleeve top": 1,
    "trousers": 2,
    "shorts": 3,
    "long sleeve top": 4,
    "skirt": 5
}

# reverse map (needed for output)
idx_to_label = {v: k for k, v in label_map.items()}


# =====================
# MODEL
# =====================
def get_model():
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,  # weights already trained
        in_channels=3,
        classes=NUM_CLASSES
    )
    return model


def load_model(model_path):
    model = get_model()
    state_dict = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.to(DEVICE)
    model.eval()
    return model


# =====================
# PREPROCESSING (MATCH TRAINING)
# =====================
def preprocess(image):
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0

    # Imagenet normalization (same as training)
    image = (image - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]

    image = np.transpose(image, (2, 0, 1)).astype(np.float32)
    image = torch.tensor(image).unsqueeze(0)

    return image.to(DEVICE)


# =====================
# POSTPROCESSING
# =====================
def get_instances(mask):
    """
    mask: (H, W) with class ids
    returns: boxes, labels, masks
    """
    boxes = []
    labels = []
    masks = []

    for cls in range(1, NUM_CLASSES):
        binary_mask = (mask == cls).astype(np.uint8)

        if binary_mask.sum() == 0:
            continue

        # connected components
        num_labels, cc = cv2.connectedComponents(binary_mask)

        for i in range(1, num_labels):  # skip background
            instance_mask = (cc == i).astype(np.uint8)

            if instance_mask.sum() < 50:  # filter tiny noise
                continue

            # bounding box
            ys, xs = np.where(instance_mask)
            x1, y1 = xs.min(), ys.min()
            x2, y2 = xs.max(), ys.max()

            boxes.append([int(x1), int(y1), int(x2), int(y2)])
            labels.append(int(cls))
            masks.append(instance_mask)

    return boxes, labels, masks


# =====================
# PREDICT
# =====================
def predict(model, image):
    """
    image: numpy array (H, W, 3) in RGB
    """

    inp = preprocess(image)

    with torch.no_grad():
        output = model(inp)
        pred_mask = torch.argmax(output, dim=1).squeeze().cpu().numpy()

    boxes, labels, masks = get_instances(pred_mask)

    # dummy scores (U-Net doesn't output confidence)
    scores = [1.0] * len(boxes)

    return {
        "boxes": boxes,
        "labels": labels,
        "scores": scores,
        "masks": masks
    }

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.8 MB/s eta 0:00:00


In [2]:
# import os
# import json
# import numpy as np
# import cv2
# import torch
# from tqdm import tqdm
# from sklearn.metrics import precision_score, recall_score, f1_score

# from predictor import load_model, predict

# # =====================
# # CONFIG
# # =====================
# IMAGE_DIR = "images"
# ANNOT_DIR = "annotations"
# SPLIT_FILE = "data_split.json"
# MODEL_PATH = "unet_tl.pth"

# IMG_SIZE = 256
# NUM_CLASSES = 6

# # =====================
# # LOAD SPLITS
# # =====================
# with open(SPLIT_FILE) as f:
#     splits = json.load(f)

# test_files = splits["test"]

# # =====================
# # LABEL MAP
# # =====================
# label_map = {
#     "short sleeve top": 1,
#     "trousers": 2,
#     "shorts": 3,
#     "long sleeve top": 4,
#     "skirt": 5
# }

# # =====================
# # MASK GENERATION (GT)
# # =====================
# def polygon_to_mask(polygons, class_id, mask):
#     for poly in polygons:
#         pts = np.array(poly).reshape(-1, 2).astype(np.int32)
#         cv2.fillPoly(mask, [pts], class_id)

# def generate_gt_mask(json_path, shape):
#     h, w = shape
#     mask = np.zeros((h, w), dtype=np.uint8)

#     with open(json_path) as f:
#         data = json.load(f)

#     for k in data:
#         if not k.startswith("item"):
#             continue

#         item = data[k]
#         cat = item["category_name"]

#         if cat not in label_map:
#             continue

#         cls = label_map[cat]
#         polygon_to_mask(item["segmentation"], cls, mask)

#     return mask

# # =====================
# # METRICS
# # =====================
# def compute_iou_per_class(pred, gt, num_classes=6):
#     ious = []
#     for cls in range(1, num_classes):
#         p = (pred == cls)
#         g = (gt == cls)

#         inter = (p & g).sum()
#         union = (p | g).sum()

#         if union == 0:
#             continue

#         ious.append(inter / union)
#     return ious

# def compute_dice_per_class(pred, gt, num_classes=6):
#     dices = []
#     for cls in range(1, num_classes):
#         p = (pred == cls)
#         g = (gt == cls)

#         inter = (p & g).sum()
#         total = p.sum() + g.sum()

#         if total == 0:
#             continue

#         dices.append((2 * inter) / total)
#     return dices

# # =====================
# # DETECTION IOU
# # =====================
# def box_iou(boxA, boxB):
#     xA = max(boxA[0], boxB[0])
#     yA = max(boxA[1], boxB[1])
#     xB = min(boxA[2], boxB[2])
#     yB = min(boxA[3], boxB[3])

#     inter = max(0, xB - xA) * max(0, yB - yA)

#     areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
#     areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

#     union = areaA + areaB - inter
#     return inter / union if union > 0 else 0

# # =====================
# # MAIN
# # =====================
# model = load_model(MODEL_PATH)

# all_ious = []
# all_dices = []

# all_gt_labels = []
# all_pred_labels = []

# tp, fp, fn = 0, 0, 0

# for img_name in tqdm(test_files):

#     img_path = os.path.join(IMAGE_DIR, img_name)
#     json_path = os.path.join(ANNOT_DIR, img_name.replace(".jpg", ".json"))

#     image = cv2.imread(img_path)
#     image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

#     h, w, _ = image.shape

#     # GT mask
#     gt_mask = generate_gt_mask(json_path, (h, w))
#     gt_mask = cv2.resize(gt_mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

#     # PREDICTION
#     out = predict(model, image)

#     pred_mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

#     for mask, label in zip(out["masks"], out["labels"]):
#         pred_mask[mask > 0] = label

#     # SEGMENTATION METRICS
#     all_ious.extend(compute_iou_per_class(pred_mask, gt_mask))
#     all_dices.extend(compute_dice_per_class(pred_mask, gt_mask))

#     # DETECTION METRICS (simplified)
#     gt_boxes = []
#     gt_labels = []

#     with open(json_path) as f:
#         data = json.load(f)

#     for k in data:
#         if not k.startswith("item"):
#             continue
#         item = data[k]
#         if item["category_name"] not in label_map:
#             continue

#         gt_boxes.append(item["bounding_box"])
#         gt_labels.append(label_map[item["category_name"]])

#     pred_boxes = out["boxes"]
#     pred_labels = out["labels"]

#     matched = set()

#     for pb, pl in zip(pred_boxes, pred_labels):
#         found = False
#         for i, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
#             if i in matched:
#                 continue
#             if pl == gl and box_iou(pb, gb) > 0.5:
#                 tp += 1
#                 matched.add(i)
#                 found = True
#                 break
#         if not found:
#             fp += 1

#     fn += len(gt_boxes) - len(matched)

#     all_gt_labels.extend(gt_labels)
#     all_pred_labels.extend(pred_labels)

# # =====================
# # FINAL METRICS
# # =====================
# mIoU = np.mean(all_ious)
# Dice = np.mean(all_dices)

# precision = tp / (tp + fp + 1e-6)
# recall = tp / (tp + fn + 1e-6)
# f1 = 2 * precision * recall / (precision + recall + 1e-6)

# print("\n===== RESULTS =====")
# print(f"mIoU: {mIoU:.4f}")
# print(f"Dice: {Dice:.4f}")
# print(f"Precision: {precision:.4f}")
# print(f"Recall: {recall:.4f}")
# print(f"F1-score: {f1:.4f}")

In [3]:
# import os
# import json
# import numpy as np
# import cv2
# import torch
# from tqdm import tqdm

# from predictor import load_model, predict

# # =====================
# # CONFIG
# # =====================
# IMAGE_DIR = "/kaggle/input/datasets/iharshsinha/deepfashion2-top5-processed/processed/train/images"
# ANNOT_DIR = "/kaggle/input/datasets/iharshsinha/deepfashion2-top5-processed/processed/train/annos"
# SPLIT_FILE = "/kaggle/input/datasets/saheemreshi/data-split-json"

# MODEL_PATHS = {
#     "scratch": "/kaggle/input/models/saheemreshi/unet-trained/pytorch/default/1/unet_scratch.pth",
#     "transfer_learning": "/kaggle/input/models/saheemreshi/unet-trained/pytorch/default/1/unet_tl.pth"
# }

# IMG_SIZE = 256
# NUM_CLASSES = 6

# # =====================
# # LOAD SPLITS
# # =====================
# with open(SPLIT_FILE) as f:
#     splits = json.load(f)

# test_files = splits["test"]

# # =====================
# # LABEL MAP
# # =====================
# label_map = {
#     "short sleeve top": 1,
#     "trousers": 2,
#     "shorts": 3,
#     "long sleeve top": 4,
#     "skirt": 5
# }

# # =====================
# # GT MASK
# # =====================
# def polygon_to_mask(polygons, class_id, mask):
#     for poly in polygons:
#         pts = np.array(poly).reshape(-1, 2).astype(np.int32)
#         cv2.fillPoly(mask, [pts], class_id)

# def generate_gt_mask(json_path, shape):
#     h, w = shape
#     mask = np.zeros((h, w), dtype=np.uint8)

#     with open(json_path) as f:
#         data = json.load(f)

#     for k in data:
#         if not k.startswith("item"):
#             continue

#         item = data[k]
#         cat = item["category_name"]

#         if cat not in label_map:
#             continue

#         cls = label_map[cat]
#         polygon_to_mask(item["segmentation"], cls, mask)

#     return mask

# # =====================
# # METRICS
# # =====================
# def compute_iou(pred, gt):
#     ious = []
#     for cls in range(1, NUM_CLASSES):
#         p = (pred == cls)
#         g = (gt == cls)

#         inter = (p & g).sum()
#         union = (p | g).sum()

#         if union == 0:
#             continue

#         ious.append(inter / union)

#     return np.mean(ious) if ious else 0


# def compute_dice(pred, gt):
#     dices = []
#     for cls in range(1, NUM_CLASSES):
#         p = (pred == cls)
#         g = (gt == cls)

#         inter = (p & g).sum()
#         total = p.sum() + g.sum()

#         if total == 0:
#             continue

#         dices.append((2 * inter) / total)

#     return np.mean(dices) if dices else 0


# def box_iou(boxA, boxB):
#     xA = max(boxA[0], boxB[0])
#     yA = max(boxA[1], boxB[1])
#     xB = min(boxA[2], boxB[2])
#     yB = min(boxA[3], boxB[3])

#     inter = max(0, xB - xA) * max(0, yB - yA)

#     areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
#     areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

#     union = areaA + areaB - inter
#     return inter / union if union > 0 else 0


# # =====================
# # EVALUATION FUNCTION
# # =====================
# def evaluate_model(model_path):

#     model = load_model(model_path)

#     iou_list = []
#     dice_list = []

#     tp, fp, fn = 0, 0, 0

#     for img_name in tqdm(test_files, desc=f"Evaluating {model_path}"):

#         img_path = os.path.join(IMAGE_DIR, img_name)
#         json_path = os.path.join(ANNOT_DIR, img_name.replace(".jpg", ".json"))

#         image = cv2.imread(img_path)
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

#         h, w, _ = image.shape

#         # GT mask
#         gt_mask = generate_gt_mask(json_path, (h, w))
#         gt_mask = cv2.resize(gt_mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

#         # Prediction
#         out = predict(model, image)

#         pred_mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
#         for mask, label in zip(out["masks"], out["labels"]):
#             pred_mask[mask > 0] = label

#         # Segmentation metrics
#         iou_list.append(compute_iou(pred_mask, gt_mask))
#         dice_list.append(compute_dice(pred_mask, gt_mask))

#         # Detection metrics
#         gt_boxes, gt_labels = [], []

#         with open(json_path) as f:
#             data = json.load(f)

#         for k in data:
#             if not k.startswith("item"):
#                 continue

#             item = data[k]
#             if item["category_name"] not in label_map:
#                 continue

#             gt_boxes.append(item["bounding_box"])
#             gt_labels.append(label_map[item["category_name"]])

#         pred_boxes = out["boxes"]
#         pred_labels = out["labels"]

#         matched = set()

#         for pb, pl in zip(pred_boxes, pred_labels):
#             found = False
#             for i, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
#                 if i in matched:
#                     continue
#                 if pl == gl and box_iou(pb, gb) > 0.5:
#                     tp += 1
#                     matched.add(i)
#                     found = True
#                     break
#             if not found:
#                 fp += 1

#         fn += len(gt_boxes) - len(matched)

#     # Final metrics
#     mIoU = np.mean(iou_list)
#     Dice = np.mean(dice_list)

#     precision = tp / (tp + fp + 1e-6)
#     recall = tp / (tp + fn + 1e-6)
#     f1 = 2 * precision * recall / (precision + recall + 1e-6)

#     return {
#         "mIoU": mIoU,
#         "Dice": Dice,
#         "Precision": precision,
#         "Recall": recall,
#         "F1": f1
#     }


# # =====================
# # RUN BOTH MODELS
# # =====================
# results = {}

# for name, path in MODEL_PATHS.items():
#     results[name] = evaluate_model(path)


# # =====================
# # PRINT COMPARISON
# # =====================
# print("\n===== FINAL COMPARISON =====")
# print(f"{'Model':<20} {'mIoU':<10} {'Dice':<10} {'Prec':<10} {'Recall':<10} {'F1':<10}")

# for name, res in results.items():
#     print(f"{name:<20} {res['mIoU']:.4f}     {res['Dice']:.4f}     {res['Precision']:.4f}     {res['Recall']:.4f}     {res['F1']:.4f}")

In [4]:
import os
import json
import numpy as np
import cv2
from tqdm import tqdm

# from predictor import load_model, predict

# =====================
# CONFIG
# =====================
IMAGE_DIR = "/kaggle/input/datasets/iharshsinha/deepfashion2-top5-processed/processed/train/images"
ANNOT_DIR = "/kaggle/input/datasets/iharshsinha/deepfashion2-top5-processed/processed/train/annos"
SPLIT_FILE = "/kaggle/input/datasets/saheemreshi/data-split-json/data_split.json"

MODEL_PATHS = {
    "scratch": "/kaggle/input/models/saheemreshi/unet-trained/pytorch/default/1/unet_scratch.pth",
    "transfer_learning": "/kaggle/input/models/saheemreshi/unet-trained/pytorch/default/1/unet_tl.pth"
}

# IMAGE_DIR = "images"
# ANNOT_DIR = "annotations"
# SPLIT_FILE = "data_split.json"

# MODEL_PATHS = {
#     "scratch": "unet_scratch.pth",
#     "transfer_learning": "unet_tl.pth"
# }

IMG_SIZE = 256
NUM_CLASSES = 6

class_names = {
    1: "short sleeve top",
    2: "trousers",
    3: "shorts",
    4: "long sleeve top",
    5: "skirt"
}

label_map = {v: k for k, v in class_names.items()}

# =====================
# LOAD SPLITS
# =====================
with open(SPLIT_FILE) as f:
    splits = json.load(f)

test_files = splits["test"]

# =====================
# GT MASK
# =====================
def polygon_to_mask(polygons, class_id, mask):
    for poly in polygons:
        pts = np.array(poly).reshape(-1, 2).astype(np.int32)
        cv2.fillPoly(mask, [pts], class_id)

def generate_gt_mask(json_path, shape):
    h, w = shape
    mask = np.zeros((h, w), dtype=np.uint8)

    with open(json_path) as f:
        data = json.load(f)

    for k in data:
        if not k.startswith("item"):
            continue

        item = data[k]
        cat = item["category_name"]

        if cat not in label_map:
            continue

        cls = label_map[cat]
        polygon_to_mask(item["segmentation"], cls, mask)

    return mask

# =====================
# IOU
# =====================
def box_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB - xA) * max(0, yB - yA)

    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    union = areaA + areaB - inter
    return inter / union if union > 0 else 0

# =====================
# EVALUATION FUNCTION
# =====================
def evaluate_model(model_path):

    model = load_model(model_path)

    iou_per_class = {i: [] for i in range(1, NUM_CLASSES)}
    dice_per_class = {i: [] for i in range(1, NUM_CLASSES)}

    tp_c = {i: 0 for i in range(1, NUM_CLASSES)}
    fp_c = {i: 0 for i in range(1, NUM_CLASSES)}
    fn_c = {i: 0 for i in range(1, NUM_CLASSES)}

    for img_name in tqdm(test_files, desc=f"Evaluating {model_path}"):

        img_path = os.path.join(IMAGE_DIR, img_name)
        json_path = os.path.join(ANNOT_DIR, img_name.replace(".jpg", ".json"))

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w, _ = image.shape

        # GT mask
        gt_mask = generate_gt_mask(json_path, (h, w))
        gt_mask = cv2.resize(gt_mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

        # Prediction
        out = predict(model, image)

        pred_mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        for mask, label in zip(out["masks"], out["labels"]):
            pred_mask[mask > 0] = label

        # =====================
        # SEGMENTATION METRICS
        # =====================
        for cls in range(1, NUM_CLASSES):
            p = (pred_mask == cls)
            g = (gt_mask == cls)

            inter = (p & g).sum()
            union = (p | g).sum()

            if union > 0:
                iou_per_class[cls].append(inter / union)

            total = p.sum() + g.sum()
            if total > 0:
                dice_per_class[cls].append((2 * inter) / total)

        # =====================
        # DETECTION METRICS
        # =====================
        gt_boxes, gt_labels = [], []

        with open(json_path) as f:
            data = json.load(f)

        for k in data:
            if not k.startswith("item"):
                continue

            item = data[k]
            cat = item["category_name"]

            if cat not in label_map:
                continue

            gt_boxes.append(item["bounding_box"])
            gt_labels.append(label_map[cat])

        pred_boxes = out["boxes"]
        pred_labels = out["labels"]

        matched = set()

        for pb, pl in zip(pred_boxes, pred_labels):
            found = False
            for i, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
                if i in matched:
                    continue
                if pl == gl and box_iou(pb, gb) > 0.5:
                    tp_c[pl] += 1
                    matched.add(i)
                    found = True
                    break
            if not found:
                fp_c[pl] += 1

        for i, gl in enumerate(gt_labels):
            if i not in matched:
                fn_c[gl] += 1

    # =====================
    # FINAL METRICS
    # =====================
    print("\n===== PER-CLASS SEGMENTATION =====")
    macro_iou, macro_dice = [], []

    for cls in range(1, NUM_CLASSES):
        iou = np.mean(iou_per_class[cls]) if iou_per_class[cls] else 0
        dice = np.mean(dice_per_class[cls]) if dice_per_class[cls] else 0

        macro_iou.append(iou)
        macro_dice.append(dice)

        print(f"{class_names[cls]:<20} IoU={iou:.4f}  Dice={dice:.4f}")

    print(f"\nMacro mIoU: {np.mean(macro_iou):.4f}")
    print(f"Macro Dice: {np.mean(macro_dice):.4f}")

    print("\n===== PER-CLASS DETECTION =====")

    for cls in range(1, NUM_CLASSES):
        tp = tp_c[cls]
        fp = fp_c[cls]
        fn = fn_c[cls]

        precision = tp / (tp + fp + 1e-6)
        recall = tp / (tp + fn + 1e-6)
        f1 = 2 * precision * recall / (precision + recall + 1e-6)

        print(f"{class_names[cls]:<20} P={precision:.4f} R={recall:.4f} F1={f1:.4f}")

    return {
        "mIoU": np.mean(macro_iou),
        "Dice": np.mean(macro_dice)
    }


# =====================
# RUN BOTH MODELS
# =====================
results = {}

for name, path in MODEL_PATHS.items():
    print(f"\n\n========== {name.upper()} ==========")
    results[name] = evaluate_model(path)


# =====================
# FINAL COMPARISON
# =====================
print("\n\n===== FINAL COMPARISON =====")
print(f"{'Model':<20} {'mIoU':<10} {'Dice':<10}")

for name, res in results.items():
    print(f"{name:<20} {res['mIoU']:.4f}     {res['Dice']:.4f}")



========== SCRATCH ==========


Evaluating /kaggle/input/models/saheemreshi/unet-trained/pytorch/default/1/unet_scratch.pth: 100%|██████████| 4326/4326 [02:58<00:00, 24.25it/s]



===== PER-CLASS SEGMENTATION =====
short sleeve top     IoU=0.4587  Dice=0.5105
trousers             IoU=0.5512  Dice=0.6219
shorts               IoU=0.4604  Dice=0.5182
long sleeve top      IoU=0.2802  Dice=0.3197
skirt                IoU=0.3494  Dice=0.3867

Macro mIoU: 0.4200
Macro Dice: 0.4714

===== PER-CLASS DETECTION =====
short sleeve top     P=0.0043 R=0.0068 F1=0.0052
trousers             P=0.0010 R=0.0012 F1=0.0011
shorts               P=0.0000 R=0.0000 F1=0.0000
long sleeve top      P=0.0010 R=0.0028 F1=0.0014
skirt                P=0.0015 R=0.0033 F1=0.0021


========== TRANSFER_LEARNING ==========


Evaluating /kaggle/input/models/saheemreshi/unet-trained/pytorch/default/1/unet_tl.pth: 100%|██████████| 4326/4326 [01:49<00:00, 39.35it/s]


===== PER-CLASS SEGMENTATION =====
short sleeve top     IoU=0.5796  Dice=0.6312
trousers             IoU=0.5833  Dice=0.6493
shorts               IoU=0.5497  Dice=0.6095
long sleeve top      IoU=0.4435  Dice=0.4894
skirt                IoU=0.4722  Dice=0.5176

Macro mIoU: 0.5257
Macro Dice: 0.5794

===== PER-CLASS DETECTION =====
short sleeve top     P=0.0052 R=0.0073 F1=0.0061
trousers             P=0.0009 R=0.0012 F1=0.0010
shorts               P=0.0000 R=0.0000 F1=0.0000
long sleeve top      P=0.0020 R=0.0028 F1=0.0023
skirt                P=0.0016 R=0.0022 F1=0.0019


===== FINAL COMPARISON =====
Model                mIoU       Dice      
scratch              0.4200     0.4714
transfer_learning    0.5257     0.5794
